# Anthropic Batch Pipeline — Conversation Generation

Generates simple two-person conversations using the Anthropic Message Batches API.

1. Build batch requests
2. Submit, poll, download
3. Post-process and filter
4. Upload to HuggingFace

In [ ]:
# 1. Create batch requests

import json
import os
import anthropic
from generate_conversations import create_conversation_prompt, iterate_params, SYSTEM_PROMPT
from conversation_data import MAX_TOKENS, topic_categories

MODEL = "claude-sonnet-4-6"
MODEL_PARAMETERS = {"top_p": 0.9}


def get_batch_requests(num_completions, offset=0):
    params_iterator = iterate_params()
    for _ in range(offset):
        next(params_iterator)

    requests = []
    for i in range(num_completions):
        params = next(params_iterator)
        system_prompt, user_prompt = create_conversation_prompt(params)

        custom_id = str(i + offset) + json.dumps(params, ensure_ascii=False)

        requests.append({
            "custom_id": custom_id,
            "params": {
                "model": MODEL,
                "max_tokens": MAX_TOKENS,
                "system": system_prompt,
                "messages": [{"role": "user", "content": user_prompt}],
                **MODEL_PARAMETERS,
            },
        })

    return requests


if not os.path.exists("data"):
    os.makedirs("data")

# Quick sanity check: print one prompt
test_requests = get_batch_requests(1)
print("Sample custom_id:", test_requests[0]["custom_id"][:80], "...")
print("\nSample prompt:")
print(test_requests[0]["params"]["messages"][0]["content"])

In [ ]:
# 2. Submit batch jobs, poll, and download results

import time
from datetime import datetime
from tqdm import tqdm

NUM_COMPLETIONS = 50_000
BATCH_SIZE = 50_000  # max requests per batch (Anthropic allows up to 100k)
OFFSET = 0
MAX_RETRIES = 3

client = anthropic.Anthropic()  # uses ANTHROPIC_API_KEY env var

directory = os.path.join("data", f"conv_batches_{datetime.now().strftime('%Y-%m-%d-%H-%M-%S')}")
os.makedirs(directory, exist_ok=True)

total_completions = 0
batch_number = 0
consecutive_failures = 0

while total_completions < NUM_COMPLETIONS and consecutive_failures < MAX_RETRIES:
    try:
        batch_number += 1
        current_offset = OFFSET + total_completions
        n = min(BATCH_SIZE, NUM_COMPLETIONS - total_completions)
        requests = get_batch_requests(n, offset=current_offset)

        print(f"Submitting batch {batch_number} with {len(requests)} requests...")
        message_batch = client.messages.batches.create(requests=requests)
        batch_id = message_batch.id

        with open(os.path.join(directory, "batch_ids.txt"), "a") as f:
            f.write(f"{batch_id}\n")

        # Poll until done
        while True:
            status = client.messages.batches.retrieve(batch_id)
            counts = status.request_counts
            print(
                f"  [{status.processing_status}] "
                f"succeeded={counts.succeeded} errored={counts.errored} "
                f"processing={counts.processing}"
            )
            if status.processing_status == "ended":
                break
            time.sleep(60)

        # Download results
        output_file = os.path.join(directory, f"batch_results_{batch_number}.jsonl")
        result_count = 0
        with open(output_file, "w", encoding="utf-8") as f:
            for result in client.messages.batches.results(batch_id):
                f.write(json.dumps(result.model_dump(), ensure_ascii=False) + "\n")
                result_count += 1

        print(f"  Downloaded {result_count} results to {output_file}")
        total_completions += n
        consecutive_failures = 0

    except Exception as e:
        print(f"Error: {e}")
        consecutive_failures += 1

if consecutive_failures >= MAX_RETRIES:
    print(f"Stopping due to {MAX_RETRIES} consecutive failures.")
else:
    print(f"Done. Generated {total_completions} conversations in {directory}")

In [ ]:
# In case you need to download results from a previous batch:
# BATCH_ID = "msgbatch_..."
# output_file = "data/conv_batches_.../batch_results_1.jsonl"
# with open(output_file, "w", encoding="utf-8") as f:
#     for result in client.messages.batches.results(BATCH_ID):
#         f.write(json.dumps(result.model_dump(), ensure_ascii=False) + "\n")

In [ ]:
# 3. Post-process batch results

import re
import json
import os
from generate_conversations import process_conversation, normalize_text, validate_conversation
from conversation_data import topic_categories

# Set this to the directory from step 2
# DIRECTORY = "conv_batches_2026-..."
# input_dir = os.path.join("data", DIRECTORY)
input_dir = directory  # use variable from cell above if running in sequence


def process_batch_file(input_file):
    results = []
    skipped = {"no_custom_id_params": 0, "not_succeeded": 0, "invalid_format": 0}

    with open(input_file, "r", encoding="utf-8") as f:
        for line in f:
            data = json.loads(line)
            custom_id = data["custom_id"]

            match = re.search(r"{.*}", custom_id)
            if not match:
                skipped["no_custom_id_params"] += 1
                continue

            params = json.loads(match.group(0))

            result = data["result"]
            if result["type"] != "succeeded":
                skipped["not_succeeded"] += 1
                continue

            completion = result["message"]["content"][0]["text"]
            gen_model = result["message"]["model"]

            processed = process_conversation(completion, params, gen_model)

            if not processed["valid"]:
                skipped["invalid_format"] += 1
                continue

            results.append(processed)

    return results, skipped


input_files = [
    os.path.join(input_dir, f)
    for f in os.listdir(input_dir)
    if f.endswith(".jsonl") and f.startswith("batch_results")
]

all_results = []
total_skipped = {}
for fpath in input_files:
    results, skipped = process_batch_file(fpath)
    all_results.extend(results)
    for k, v in skipped.items():
        total_skipped[k] = total_skipped.get(k, 0) + v

output_file = os.path.join(input_dir, "processed.jsonl")
with open(output_file, "w", encoding="utf-8") as f:
    for r in all_results:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"Processed {len(all_results)} valid conversations")
print(f"Skipped: {total_skipped}")

In [ ]:
# 3.1 Filtering and analysis

import pandas as pd
from unidecode import unidecode

df = pd.read_json(output_file, lines=True)
print(f"Total conversations before filtering: {len(df)}")

# Normalize to ASCII
df["conversation"] = df["conversation"].apply(lambda x: unidecode(x))

# Filter by character count
df = df[df["character_count"] >= 1024]
df = df[df["character_count"] <= 2048]
print(f"After character count filter (1024-2048): {len(df)}")

# Filter by minimum turns
df = df[df["total_turns"] >= 3]
print(f"After minimum turns filter (>=3): {len(df)}")

# Clean up word type display
df["initial_word_type"] = df["initial_word_type"].apply(
    lambda x: x.split()[-1] if x else x
)

# Drop internal columns
df = df.drop(columns=["valid", "errors", "generation_id"], errors="ignore")

# Shuffle
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Reorder columns
ordered_cols = [
    "conversation", "topic", "topic_category", "tone", "mode",
    "grammar", "initial_word_type", "initial_letter",
    "word_count", "character_count", "person1_turns", "person2_turns", "total_turns",
    "model",
]
df = df[[c for c in ordered_cols if c in df.columns]]

print(f"\nFinal dataset: {len(df)} conversations")
print(f"\nMode distribution:")
print(df["mode"].value_counts(normalize=True))
print(f"\nTopic category distribution:")
print(df["topic_category"].value_counts())
print(f"\nCharacter count stats:")
print(df["character_count"].describe())

In [ ]:
# 3.2 Inspect a few samples

for i in range(min(5, len(df))):
    row = df.iloc[i]
    print("=" * 60)
    print(f"Topic: {row['topic']} ({row['topic_category']}) | Mode: {row['mode']} | Tone: {row['tone']}")
    print(f"Chars: {row['character_count']} | Turns: {row['total_turns']}")
    print("-" * 60)
    print(row["conversation"][:500])
    print("..." if len(row["conversation"]) > 500 else "")
    print()

In [ ]:
# 4. Upload to HuggingFace
# Run: huggingface-cli login  (if not already logged in)

from datasets import Dataset

dataset = Dataset.from_pandas(df)
dataset = dataset.train_test_split(test_size=0.01, seed=42)
dataset.push_to_hub("your-username/SimpleConversations")